# 03_eda_final — Learning Behavior Analytics & At-Risk Learner EDA

This notebook is a refactored EDA blueprint for the OULAD-based final project.

**Notebook contract**
- Preserve the native analytical grain: **student–module–presentation**
- Separate **descriptive full-course outcome analysis** from **early-warning analysis**
- Keep the two-part story:
  1. learning behavior analytics for the full learner population
  2. deep-dive on at-risk learners
- Export reusable tables for **feature engineering, segmentation, at-risk modeling, report, slide, and Power BI**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid')

RAW = Path('../data/raw')
PROC = Path('../data/processed')
FIG_DIR = Path('../figures/eda')
TBL_DIR = Path('../tables/eda_summary')
BI_DIR = Path('../data/processed/eda_outputs')

for p in [FIG_DIR, TBL_DIR, BI_DIR]:
    p.mkdir(parents=True, exist_ok=True)

KEYS = ['id_student', 'code_module', 'code_presentation']
WINDOWS = [7, 14, 21, 28]

def save_fig(name: str):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'{name}.png', dpi=160, bbox_inches='tight')

def save_csv(df: pd.DataFrame, name: str):
    df.to_csv(BI_DIR / f'{name}.csv', index=False)

def pct(x):
    return round(x * 100, 2)

## 0. Load tables and rebuild the correct analytical layer

This project should not rely on a student-level collapse of `studentInfo`.  
The correct base table for EDA is **one row per student–module–presentation**.

In [ ]:
# Prefer corrected processed files if they exist; otherwise rebuild from raw OULAD tables
student_info = pd.read_csv(RAW / 'studentInfo.csv')
student_reg = pd.read_csv(RAW / 'studentRegistration.csv')
assessments = pd.read_csv(RAW / 'assessments.csv')
student_assessment = pd.read_csv(RAW / 'studentAssessment.csv')
vle = pd.read_csv(RAW / 'vle.csv')
student_vle = pd.read_csv(RAW / 'studentVle.csv')
courses = pd.read_csv(RAW / 'courses.csv')

student_info = student_info.drop_duplicates(subset=KEYS).copy()
student_info['is_at_risk'] = student_info['final_result'].isin(['Withdrawn', 'Fail']).astype('int8')

master_mp = (
    student_info
    .merge(student_reg, on=KEYS, how='left', validate='1:1')
    .merge(courses, on=['code_module', 'code_presentation'], how='left', validate='m:1')
)

assessment_enriched = (
    student_assessment
    .merge(assessments, on='id_assessment', how='left', validate='m:1')
    .merge(master_mp[KEYS + ['final_result', 'is_at_risk']], on=KEYS, how='left', validate='m:1')
)
assessment_enriched['submission_delay'] = assessment_enriched['date_submitted'] - assessment_enriched['date']
assessment_enriched['is_late'] = (assessment_enriched['submission_delay'] > 0).astype('int8')

vle_enriched = (
    student_vle
    .merge(
        vle[['id_site', 'code_module', 'code_presentation', 'activity_type', 'week_from', 'week_to']],
        on=['id_site', 'code_module', 'code_presentation'],
        how='left',
        validate='m:1'
    )
    .merge(master_mp[KEYS + ['final_result', 'is_at_risk']], on=KEYS, how='left', validate='m:1')
)
vle_enriched['week'] = np.floor_divide(vle_enriched['date'], 7).astype(int)

display(master_mp.head())

## 1. Executive data snapshot
**Goal**: establish dataset footprint, learner/outcome mix, and the binary at-risk frame.

In [ ]:
snapshot = pd.DataFrame({
    'dataset': ['student_info', 'student_registration', 'assessments', 'student_assessment', 'vle', 'student_vle', 'master_mp'],
    'rows': [len(student_info), len(student_reg), len(assessments), len(student_assessment), len(vle), len(student_vle), len(master_mp)],
    'grain': [
        'student-module-presentation',
        'student-module-presentation',
        'assessment',
        'submission',
        'site-module-presentation',
        'student-site-day',
        'student-module-presentation'
    ]
})
display(snapshot)
save_csv(snapshot, 'eda_exec_snapshot')

outcome_dist = (
    master_mp['final_result']
    .value_counts(dropna=False)
    .rename_axis('final_result')
    .reset_index(name='n')
)
outcome_dist['pct'] = outcome_dist['n'] / outcome_dist['n'].sum()
save_csv(outcome_dist, 'outcome_distribution_mp')

plt.figure(figsize=(8, 4))
sns.barplot(data=outcome_dist, x='final_result', y='n')
plt.title('Outcome distribution at student-module-presentation grain')
save_fig('01_outcome_distribution')
plt.show()

## 2. Data quality, grain validation, and coverage
**Goal**: prove that joins respect keys, avoid silent duplication, and quantify coverage.

In [ ]:
def uniqueness_check(df, keys, name):
    return {
        'table': name,
        'rows': len(df),
        'unique_keys': df[keys].drop_duplicates().shape[0],
        'duplicate_key_rows': len(df) - df[keys].drop_duplicates().shape[0]
    }

grain_checks = pd.DataFrame([
    uniqueness_check(student_info, KEYS, 'student_info'),
    uniqueness_check(student_reg, KEYS, 'student_registration'),
    uniqueness_check(master_mp, KEYS, 'master_mp')
])
display(grain_checks)
save_csv(grain_checks, 'grain_checks')

coverage = master_mp[KEYS + ['final_result']].copy()
coverage['has_vle'] = coverage.set_index(KEYS).index.isin(vle_enriched[KEYS].drop_duplicates().set_index(KEYS).index)
coverage['has_assessment'] = coverage.set_index(KEYS).index.isin(assessment_enriched[KEYS].drop_duplicates().set_index(KEYS).index)

coverage_summary = pd.DataFrame({
    'metric': ['rows', 'has_vle', 'has_assessment'],
    'value': [len(coverage), coverage['has_vle'].mean(), coverage['has_assessment'].mean()]
})
display(coverage_summary)
save_csv(coverage_summary, 'coverage_summary')

## 3. Outcome framework
**Goal**: keep the four-class outcome story while defining the binary at-risk label for later use.

In [ ]:
master_mp['success_flag'] = master_mp['final_result'].isin(['Pass', 'Distinction']).astype('int8')
master_mp['risk_type'] = np.select(
    [
        master_mp['final_result'].eq('Withdrawn'),
        master_mp['final_result'].eq('Fail')
    ],
    ['Withdrawn', 'Fail'],
    default='Not at risk'
)

outcome_framework = (
    master_mp.groupby(['code_module', 'code_presentation'])['final_result']
    .value_counts(normalize=True)
    .rename('share')
    .reset_index()
)
save_csv(outcome_framework, 'outcome_framework_by_presentation')
display(outcome_framework.head())

## 4. Learner distribution and profile
**Goal**: profile the population by age, education, credits, prior attempts, disability, and region.

In [ ]:
profile_summary = (
    master_mp.groupby(['final_result', 'highest_education'])
    .agg(
        enrollments=('id_student', 'count'),
        mean_prev_attempts=('num_of_prev_attempts', 'mean'),
        mean_credits=('studied_credits', 'mean')
    )
    .reset_index()
)
save_csv(profile_summary, 'learner_profile_summary')
display(profile_summary.head())

subgroup_risk = (
    master_mp.groupby(['highest_education', 'disability'])['is_at_risk']
    .mean()
    .rename('at_risk_rate')
    .reset_index()
)
display(subgroup_risk)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=master_mp.groupby('highest_education', as_index=False)['is_at_risk'].mean().sort_values('is_at_risk', ascending=False),
    x='is_at_risk', y='highest_education'
)
plt.xlabel('At-risk rate')
plt.ylabel('Highest education')
plt.title('At-risk rate by education level')
save_fig('02_risk_by_education')
plt.show()

## 5. Module and presentation heterogeneity
**Goal**: quantify how strongly outcomes differ by course context.

In [ ]:
mp_summary = (
    master_mp.groupby(['code_module', 'code_presentation'])
    .agg(
        enrollments=('id_student', 'count'),
        withdrawn_rate=('final_result', lambda s: (s == 'Withdrawn').mean()),
        fail_rate=('final_result', lambda s: (s == 'Fail').mean()),
        at_risk_rate=('is_at_risk', 'mean')
    )
    .reset_index()
)
save_csv(mp_summary, 'module_presentation_risk_gradient')
display(mp_summary.head())

pivot_risk = mp_summary.pivot(index='code_module', columns='code_presentation', values='at_risk_rate')
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_risk, annot=True, fmt='.2f', cmap='YlOrRd')
plt.title('At-risk rate by module and presentation')
save_fig('03_module_presentation_risk_heatmap')
plt.show()

## 6. Engagement behavior over time
**Goal**: study temporal engagement at the student-week level rather than raw interaction-record means.

In [ ]:
student_week = (
    vle_enriched[vle_enriched['date'] >= 0]
    .groupby(KEYS + ['final_result', 'week'])
    .agg(
        weekly_clicks=('sum_click', 'sum'),
        active_days=('date', 'nunique'),
        unique_sites=('id_site', 'nunique')
    )
    .reset_index()
)

weekly_outcome = (
    student_week.groupby(['final_result', 'week'])
    .agg(
        median_clicks=('weekly_clicks', 'median'),
        mean_active_days=('active_days', 'mean'),
        learner_weeks=('id_student', 'count')
    )
    .reset_index()
)
save_csv(weekly_outcome, 'weekly_engagement_outcome')

plt.figure(figsize=(10, 5))
sns.lineplot(data=weekly_outcome[weekly_outcome['week'].between(0, 20)], x='week', y='median_clicks', hue='final_result')
plt.title('Median weekly clicks by final result')
save_fig('04_weekly_median_clicks_by_outcome')
plt.show()

## 7. Learning material usage and assessment behavior
**Goal**: move from engagement volume to engagement composition and submission discipline.

In [ ]:
activity_mix = (
    vle_enriched[vle_enriched['date'] >= 0]
    .groupby(['final_result', 'activity_type'])['sum_click']
    .sum()
    .rename('total_clicks')
    .reset_index()
)
activity_mix['click_share'] = activity_mix.groupby('final_result')['total_clicks'].transform(lambda s: s / s.sum())
save_csv(activity_mix, 'activity_mix_by_outcome')

assessment_summary = (
    assessment_enriched.groupby(['final_result', 'assessment_type'])
    .agg(
        n_submissions=('id_assessment', 'count'),
        avg_score=('score', 'mean'),
        median_score=('score', 'median'),
        late_rate=('is_late', 'mean'),
        avg_delay=('submission_delay', 'mean')
    )
    .reset_index()
)
save_csv(assessment_summary, 'assessment_behavior_by_outcome')
display(assessment_summary.head())

## 8. Outcome comparison by final_result
**Goal**: synthesize multiple behavior dimensions into one comparative view.

In [ ]:
full_course_features = (
    student_week.groupby(KEYS)
    .agg(
        total_clicks=('weekly_clicks', 'sum'),
        active_weeks=('week', 'nunique'),
        mean_weekly_clicks=('weekly_clicks', 'mean'),
        click_volatility=('weekly_clicks', 'std')
    )
    .reset_index()
    .merge(
        assessment_enriched.groupby(KEYS).agg(
            avg_score=('score', 'mean'),
            submissions=('id_assessment', 'count'),
            late_rate=('is_late', 'mean'),
            avg_delay=('submission_delay', 'mean')
        ).reset_index(),
        on=KEYS,
        how='left'
    )
    .fillna(0)
)

outcome_metric_profile = (
    master_mp[KEYS + ['final_result']]
    .merge(full_course_features, on=KEYS, how='left')
    .groupby('final_result')[['total_clicks', 'active_weeks', 'avg_score', 'submissions', 'late_rate', 'click_volatility']]
    .mean()
)

outcome_metric_z = (outcome_metric_profile - outcome_metric_profile.mean()) / outcome_metric_profile.std()
save_csv(outcome_metric_profile.reset_index(), 'final_result_metric_profile')

plt.figure(figsize=(10, 4))
sns.heatmap(outcome_metric_z, annot=True, fmt='.2f', cmap='RdYlGn')
plt.title('Standardized metric profile by final result')
save_fig('05_metric_profile_heatmap')
plt.show()

## 9. Early-warning exploratory analytics (7/14/21/28-day windows)
**Goal**: create leakage-safe descriptive signals for later at-risk modeling.

In [ ]:
def build_window_features(days: int) -> pd.DataFrame:
    vle_w = (
        vle_enriched[vle_enriched['date'].between(0, days)]
        .groupby(KEYS)
        .agg(
            clicks=('sum_click', 'sum'),
            active_days=('date', 'nunique'),
            activity_breadth=('activity_type', 'nunique'),
            last_active_day=('date', 'max')
        )
        .reset_index()
    )

    assess_w = (
        assessment_enriched[assessment_enriched['date_submitted'].between(0, days)]
        .groupby(KEYS)
        .agg(
            early_submissions=('id_assessment', 'count'),
            early_avg_score=('score', 'mean'),
            early_late_rate=('is_late', 'mean')
        )
        .reset_index()
    )

    due_counts = (
        assessments[assessments['date'].le(days)]
        .groupby(['code_module', 'code_presentation'])
        .size()
        .rename('assessments_due_by_window')
        .reset_index()
    )

    window_df = (
        master_mp[KEYS + ['final_result', 'is_at_risk']]
        .merge(vle_w, on=KEYS, how='left')
        .merge(assess_w, on=KEYS, how='left')
        .merge(due_counts, on=['code_module', 'code_presentation'], how='left')
        .fillna({
            'clicks': 0,
            'active_days': 0,
            'activity_breadth': 0,
            'last_active_day': -1,
            'early_submissions': 0,
            'early_avg_score': 0,
            'early_late_rate': 0,
            'assessments_due_by_window': 0
        })
    )
    window_df['missing_due_assessments'] = window_df['assessments_due_by_window'] - window_df['early_submissions']
    window_df['window_days'] = days
    return window_df

window_features = {d: build_window_features(d) for d in WINDOWS}

for d, df_w in window_features.items():
    save_csv(df_w, f'early_window_features_{d}d')

window_signal_summary = pd.concat([
    df.groupby('final_result')[['clicks', 'active_days', 'early_submissions', 'early_avg_score', 'early_late_rate']]
      .mean()
      .assign(window_days=days)
      .reset_index()
    for days, df in window_features.items()
], ignore_index=True)

save_csv(window_signal_summary, 'early_signal_summary')
display(window_signal_summary.head())

## 10. At-risk learner profiling and risk decomposition
**Goal**: separate Withdrawn from Fail, analyze withdrawal timing, and identify practical risk gradients.

In [ ]:
withdrawal_timing = (
    master_mp[master_mp['final_result'] == 'Withdrawn']
    .assign(unregister_week=lambda d: np.floor_divide(d['date_unregistration'], 7))
    .groupby(['code_module', 'code_presentation', 'unregister_week'])
    .size()
    .rename('n_withdrawn')
    .reset_index()
)
save_csv(withdrawal_timing, 'withdrawal_timing_by_presentation')

risk_subgroup_heatmap = (
    master_mp.assign(prev_attempt_band=pd.cut(master_mp['num_of_prev_attempts'], bins=[-1, 0, 1, 10], labels=['0', '1', '2+']))
    .groupby(['prev_attempt_band', 'highest_education'])['is_at_risk']
    .mean()
    .reset_index()
)
display(risk_subgroup_heatmap.head())

## 11. EDA synthesis for downstream work
**Goal**: explicitly map EDA outputs to feature engineering, segmentation, modeling, recommendation, and Power BI.

In [ ]:
risk_feature_candidates = pd.DataFrame({
    'feature_name': [
        'early_clicks_7d', 'early_clicks_14d', 'active_days_14d', 'activity_breadth_28d',
        'early_submissions_28d', 'early_avg_score_28d', 'early_late_rate_28d',
        'click_volatility', 'active_weeks', 'days_to_last_activity'
    ],
    'eda_rationale': [
        'early engagement intensity',
        'engagement persistence',
        'behavior regularity',
        'resource breadth',
        'assessment participation',
        'early academic signal',
        'submission discipline',
        'stability/volatility',
        'consistency over the course',
        'recency/possible disengagement'
    ],
    'safe_for_early_model': [True, True, True, True, True, True, True, False, False, True]
})
save_csv(risk_feature_candidates, 'risk_feature_candidates')
display(risk_feature_candidates)

## Final notebook checklist
- Every analysis respects **student–module–presentation** grain
- Early-warning sections use only data available up to the cutoff window
- Full-course analyses are clearly labeled as **descriptive end-state analysis**
- Every figure has one sentence of interpretation and one sentence of downstream relevance
- Final CSV exports are ready for report, slide, and Power BI